# Module 00, Unit 03 — Python Lab
## Single-Variable Review & Notation Setup

> **Threat Surfaces: Multivariable Calculus for AI Security**  
> fischer³ Education | Module 00 | Unit 03
>
> **Estimated time**: 15–20 minutes  
> **Prerequisites**: Complete `notes.md` and `exercises.tex` first

---

### What This Lab Does

This lab has three sections:

1. **Verify your derivatives** — Plot functions and their derivatives side by side, confirming your Exercise 1 answers visually and numerically.
2. **Visualize the log-likelihood surface** — Plot $\ell(\mu)$ for the Normal model, see the score function, and confirm the MLE numerically.
3. **The Gaussian integral** — Explore the integral $\int_{-\infty}^{\infty} e^{-x^2/2}\, dx$ numerically before a preview of its closed form.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import integrate

# Course colour palette (matches LaTeX preamble)
TS_BLUE   = '#1E64B4'
TS_AMBER  = '#C87814'
TS_GREEN  = '#1E8C50'
TS_GRAY   = '#64646E'
TS_LIGHT  = '#F5F7FA'

plt.rcParams.update({
    'figure.facecolor': TS_LIGHT,
    'axes.facecolor':   TS_LIGHT,
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Setup complete.')

---
## Section 1 — Verify Your Derivatives

We will plot each function from Exercise 1 alongside its derivative.  
A numerical derivative (using a finite difference) provides an independent check of your analytical result.

**Before running the cell below**, write your answers to Exercise 1 parts (a)–(c) here:

- $f(x) = e^{-3x^2 + 2x}$ → $f'(x) = $ *(your answer)*
- $g(x) = x^3 \ln(x^2 + 1)$ → $g'(x) = $ *(your answer)*
- $h(x) = \ln\!\left(\frac{e^x}{1+e^x}\right)$ → $h'(x) = $ *(your answer)*

If the plotted analytical and numerical derivatives agree, your answers are correct.

In [ ]:
def numerical_deriv(f, x, h=1e-5):
    """Central finite difference approximation to f'(x)."""
    return (f(x + h) - f(x - h)) / (2 * h)

x = np.linspace(-2, 2, 400)

# ── Define the three functions ──────────────────────────────────────────────
f = lambda x: np.exp(-3*x**2 + 2*x)
g = lambda x: x**3 * np.log(x**2 + 1)
h = lambda x: np.log(np.exp(x) / (1 + np.exp(x)))   # = log(sigmoid(x))

# ── Fill in your analytical derivatives here ────────────────────────────────
# Replace the lambda bodies with your Exercise 1 answers.
# The cell will raise an error if your derivative does not match numerically.

df_analytical = lambda x: np.exp(-3*x**2 + 2*x) * (-6*x + 2)    # SOLUTION — fill in yours first
dg_analytical = lambda x: 3*x**2 * np.log(x**2 + 1) + x**3 * (2*x / (x**2 + 1))
dh_analytical = lambda x: 1 / (1 + np.exp(x))   # sigmoid identity from notes

functions   = [f,  g,  h]
analyticals = [df_analytical, dg_analytical, dh_analytical]
labels      = [
    r'$f(x) = e^{-3x^2+2x}$',
    r'$g(x) = x^3 \ln(x^2+1)$',
    r'$h(x) = \ln\!\left(\frac{e^x}{1+e^x}\right)$',
]

fig, axes = plt.subplots(3, 2, figsize=(11, 12))
fig.suptitle('Exercise 1 — Functions and Their Derivatives', fontsize=13, color=TS_GRAY)

for i, (func, anal, label) in enumerate(zip(functions, analyticals, labels)):
    # Left column: the function
    axes[i, 0].plot(x, func(x), color=TS_BLUE, lw=2)
    axes[i, 0].set_title(label, color=TS_GRAY)
    axes[i, 0].axhline(0, color=TS_GRAY, lw=0.5, ls='--')

    # Right column: analytical vs numerical derivative
    num_d = numerical_deriv(func, x)
    axes[i, 1].plot(x, anal(x),  color=TS_BLUE,  lw=2,   label='Analytical')
    axes[i, 1].plot(x, num_d,    color=TS_AMBER, lw=1.5, ls='--', label='Numerical')
    axes[i, 1].set_title("Derivative", color=TS_GRAY)
    axes[i, 1].legend(fontsize=9)
    axes[i, 1].axhline(0, color=TS_GRAY, lw=0.5, ls='--')

    # Verify agreement (max relative error < 0.1%)
    max_err = np.max(np.abs(anal(x) - num_d))
    print(f"Part {'(a b c)'[i*2]}: max absolute error = {max_err:.2e}", 
          '✓' if max_err < 1e-3 else '✗  — check your derivative')

plt.tight_layout()
plt.show()

> **What Do You See?**  
> For each function, identify: (1) where the derivative is zero (what does this mean geometrically?), and (2) where the derivative changes sign (what happens to the function at those points)?

---
## Section 2 — The Log-Likelihood Surface

We will work with the Normal log-likelihood from Exercise 3 and Problem~3 of the exercises:

$$\ell(\mu) = -\frac{n}{2}\ln(2\pi) - \frac{1}{2}\sum_{i=1}^n (x_i - \mu)^2$$

We will:
- Generate a small sample from a known distribution
- Plot $\ell(\mu)$ as a function of $\mu$
- Overlay the score function $\ell'(\mu)$ and mark where it crosses zero
- Confirm the MLE equals the sample mean

In [ ]:
rng = np.random.default_rng(seed=42)

TRUE_MU    = 3.0
TRUE_SIGMA = 1.0
N          = 20

data = rng.normal(loc=TRUE_MU, scale=TRUE_SIGMA, size=N)
x_bar = data.mean()

print(f"True μ:       {TRUE_MU:.3f}")
print(f"Sample mean:  {x_bar:.3f}  ← this should be the MLE")
print(f"n = {N} observations")

In [ ]:
mu_grid = np.linspace(TRUE_MU - 3, TRUE_MU + 3, 500)

def log_likelihood(mu, data, sigma=1.0):
    n = len(data)
    return -(n / 2) * np.log(2 * np.pi * sigma**2) - (1 / (2 * sigma**2)) * np.sum((data - mu)**2)

def score(mu, data, sigma=1.0):
    """Derivative of log-likelihood with respect to mu."""
    # From Exercise 3 part (a) — fill in your answer, then uncomment:
    # return ...
    return np.sum(data - mu) / sigma**2   # SOLUTION

ll_values    = np.array([log_likelihood(mu, data) for mu in mu_grid])
score_values = np.array([score(mu, data) for mu in mu_grid])

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(r'Normal Log-Likelihood: $\sigma^2 = 1$, $n = 20$',
             fontsize=13, color=TS_GRAY)

# Left: log-likelihood
ax1.plot(mu_grid, ll_values, color=TS_BLUE, lw=2)
ax1.axvline(x_bar,    color=TS_AMBER, lw=2, ls='--', label=f'MLE = $\\bar{{x}}$ = {x_bar:.2f}')
ax1.axvline(TRUE_MU,  color=TS_GREEN, lw=1.5, ls=':', label=f'True μ = {TRUE_MU}')
ax1.set_xlabel(r'$\mu$')
ax1.set_ylabel(r'$\ell(\mu)$')
ax1.set_title('Log-Likelihood Surface')
ax1.legend()

# Right: score function
ax2.plot(mu_grid, score_values, color=TS_BLUE, lw=2)
ax2.axhline(0,        color=TS_GRAY,  lw=0.8, ls='--')
ax2.axvline(x_bar,    color=TS_AMBER, lw=2,   ls='--', label=f'Score = 0 at $\\bar{{x}}$ = {x_bar:.2f}')
ax2.set_xlabel(r'$\mu$')
ax2.set_ylabel(r"$\ell'(\mu)$")
ax2.set_title('Score Function (Derivative of Log-Likelihood)')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nMLE = sample mean = {x_bar:.4f}")
print(f"Log-likelihood at MLE:    {log_likelihood(x_bar, data):.4f}")
print(f"Score at MLE (should ≈ 0): {score(x_bar, data):.2e}")

> **What Do You See?**
> 1. The log-likelihood surface is a downward-opening parabola in $\mu$. Why? (Look at the formula — what kind of function is it?)
> 2. The score crosses zero exactly at the sample mean. This is the MLE condition from Exercise 3. Does the score change sign? What does that confirm?
> 3. The MLE is close to but not equal to the true $\mu = 3$. Why is this expected? What would happen if you increased $n$?

In [ ]:
# ── Extension challenge ───────────────────────────────────────────────────────
# Plot the log-likelihood for several sample sizes: n = 5, 20, 100, 500.
# Normalise by n so they are comparable on the same axis.
# What happens to the curvature as n grows?
# (The negative of the curvature is related to the Fisher information — a concept
# you will formalise in Module 02.)

ns = [5, 20, 100, 500]
fig, ax = plt.subplots(figsize=(9, 5))
colors = [TS_BLUE, TS_AMBER, TS_GREEN, TS_GRAY]

for n_val, col in zip(ns, colors):
    sample = rng.normal(loc=TRUE_MU, scale=TRUE_SIGMA, size=n_val)
    ll_norm = np.array([log_likelihood(mu, sample) / n_val for mu in mu_grid])
    ax.plot(mu_grid, ll_norm, color=col, lw=2, label=f'$n = {n_val}$')

ax.axvline(TRUE_MU, color='black', lw=1, ls=':', label=f'True μ = {TRUE_MU}')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\ell(\mu) / n$')
ax.set_title('Normalised Log-Likelihood for Increasing Sample Sizes')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 3 — The Gaussian Integral

Exercise 2(c) asked you to state the result $\int_{-\infty}^{\infty} e^{-x^2/2}\, dx = \sqrt{2\pi}$ without deriving it.

Here we:
- Verify this numerically using `scipy.integrate.quad`
- Visualize the integrand and the area it accumulates
- Preview the normalisation argument that makes the Gaussian PDF integrate to 1

In [ ]:
# Numerical integration
integrand = lambda x: np.exp(-x**2 / 2)
result, error = integrate.quad(integrand, -np.inf, np.inf)

print(f"∫ e^(-x²/2) dx  =  {result:.8f}")
print(f"√(2π)           =  {np.sqrt(2 * np.pi):.8f}")
print(f"Absolute error  =  {error:.2e}")

# ── Visualise accumulation ────────────────────────────────────────────────────
x_plot = np.linspace(-5, 5, 600)
y_plot = np.exp(-x_plot**2 / 2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(r'The Gaussian Integral: $\int_{-\infty}^{\infty} e^{-x^2/2}\, dx = \sqrt{2\pi}$',
             fontsize=13, color=TS_GRAY)

# Left: the integrand
ax1.fill_between(x_plot, y_plot, alpha=0.25, color=TS_BLUE)
ax1.plot(x_plot, y_plot, color=TS_BLUE, lw=2)
ax1.set_xlabel('$x$')
ax1.set_ylabel(r'$e^{-x^2/2}$')
ax1.set_title('Integrand')
ax1.text(0, 0.5, f'Area = {result:.4f} = √(2π)', ha='center', color=TS_GRAY)

# Right: the normalised PDF (divide by √(2π))
pdf = y_plot / np.sqrt(2 * np.pi)
ax2.fill_between(x_plot, pdf, alpha=0.25, color=TS_GREEN)
ax2.plot(x_plot, pdf, color=TS_GREEN, lw=2)
ax2.set_xlabel('$x$')
ax2.set_ylabel(r'$\phi(x)$')
ax2.set_title(r'Standard Normal PDF: $\phi(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2}$')

pdf_integral, _ = integrate.quad(lambda x: np.exp(-x**2/2) / np.sqrt(2*np.pi), -np.inf, np.inf)
ax2.text(0, 0.25, f'Area = {pdf_integral:.6f}', ha='center', color=TS_GRAY)

plt.tight_layout()
plt.show()

> **What Do You See?**
> 1. The integrand $e^{-x^2/2}$ has area $\sqrt{2\pi} \approx 2.507$. Dividing by $\sqrt{2\pi}$ normalises the area to 1. Why must any valid probability density integrate to 1?
> 2. The PDF decays rapidly away from 0. Approximately what fraction of the total area lies within $|x| \leq 2$? Use `integrate.quad` to find out.

In [ ]:
# Your code here: compute the fraction of the standard Normal within |x| ≤ 2


---
## Wrap-Up

**What this lab confirmed:**
- Analytical derivatives can be verified numerically — if they disagree, recheck your algebra.
- The MLE for a Normal model is the sample mean. Setting the score function to zero recovers it exactly.
- The Gaussian integral $\sqrt{2\pi}$ is the normalising constant that makes the Normal PDF integrate to 1.

**What comes next:**
Module 01 builds the geometric intuition — scalar fields, level sets, and the visual structure of multivariable functions — that will make the generalisation to gradients and Hessians feel natural rather than abstract.

---
*Module 00, Unit 03 — Threat Surfaces, fischer³ Education*